# Measuring Uncertainty in Finite Samples

The first statement gives a large-sample normal approximation; the second gives the simulation error of an average:

$$
\sqrt n\,(\widehat\theta_n-\theta)\Rightarrow\mathcal N(0,V),
\qquad
\operatorname{SE}(\overline Z_M)=\frac{s_Z}{\sqrt M}.
$$

Here, $n$ is sample size; $\widehat\theta_n$ estimates $\theta$; $V$ is the limiting variance; $\Rightarrow$ denotes convergence in distribution. $M$ is the number of simulation runs; $\overline Z_M$ is their average; $s_Z$ is their sample standard deviation; $\operatorname{SE}$ is standard error.

In [1]:
import re as _table_number_re
from IPython.display import HTML as _TableHTML, display as _raw_display


def compact_float_format(number_format=None):
    def format_value(value):
        if number_format is None:
            rendered = str(value)
        elif callable(number_format):
            rendered = str(number_format(value))
        else:
            rendered = number_format % value
        return compact_numeric_token(rendered)
    return format_value


def compact_numeric_token(rendered):
    if not _table_number_re.fullmatch(r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?", rendered):
        return rendered
    value = float(rendered)
    if value != 0.0 and abs(value) < 5e-13:
        return "0"
    if ("." in rendered or "e" in rendered.lower()) and value.is_integer():
        return str(int(value))
    return rendered


def compact_numeric_text(text):
    pattern = r"(?<![0-9.eE])([+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?)(?![0-9.eE])"
    return _table_number_re.sub(pattern, lambda match: compact_numeric_token(match.group(0)), text)


def compact_table_latex(latex):
    return compact_numeric_text(latex)


def _compact_table_html(value):
    frame = value.to_frame() if isinstance(value, pd.Series) else value
    html = frame._repr_html_()
    return '<div style="display:flex;justify-content:center;">' + compact_numeric_text(html) + "</div>"


def display(value):
    if isinstance(value, (pd.DataFrame, pd.Series)):
        return _raw_display(_TableHTML(_compact_table_html(value)))
    return _raw_display(value)

from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = Path.cwd().parents[1]
import RiskLabAI

assert RiskLabAI.__version__ == "3.2.0"

from RiskLabAI.data.synthetic_data.simulation import form_true_matrix, simulates_cov_mu

FIGURES = ROOT / "outputs" / "figures" / "01_mathematical_language"
TABLES = ROOT / "outputs" / "tables" / "01_mathematical_language"
FIGURE_DATA = ROOT / "outputs" / "figure_data" / "01_mathematical_language"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
FIGURE_DATA.mkdir(parents=True, exist_ok=True)

palette = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "figure.facecolor": palette["background"],
    "axes.facecolor": palette["axes"],
    "axes.edgecolor": palette["text"],
    "axes.labelcolor": palette["text"],
    "axes.titlecolor": palette["text"],
    "xtick.color": palette["text"],
    "ytick.color": palette["text"],
    "text.color": palette["text"],
    "grid.color": palette["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": palette["axes"],
    "legend.edgecolor": palette["grid"],
    "savefig.facecolor": palette["background"],
})

## Monte Carlo Design

The equations estimate average error and root mean squared error across repeated samples:

$$
\widehat b_M=\frac{1}{M}\sum_{m=1}^{M}(\widehat\theta^{(m)}-\theta),
\qquad
\widehat{\operatorname{RMSE}}_M=
\left[\frac{1}{M}\sum_{m=1}^{M}\lVert\widehat\theta^{(m)}-\theta\rVert^2\right]^{1/2}.
$$

Here, $M$ is the number of repetitions; $m$ labels a repetition; $\widehat\theta^{(m)}$ is its estimate; $\theta$ is the true target. $\widehat b_M$ is estimated bias; $\widehat{\operatorname{RMSE}}_M$ is estimated root mean squared error; $\lVert\cdot\rVert$ is a vector or matrix norm.

In [2]:
def minimum_variance_weights(covariance):
    unit = np.ones(covariance.shape[0])
    solution = np.linalg.solve(covariance, unit)
    return solution / (unit @ solution)

np.random.seed(20260925)
population_mean, population_covariance = form_true_matrix(
    n_blocks=3,
    block_size=2,
    block_correlation=0.65,
)
population_weights = minimum_variance_weights(population_covariance)
minimum_risk = population_weights @ population_covariance @ population_weights
sample_sizes = np.array([40, 80, 160, 320, 640, 1280])
repetitions = 240
np.random.seed(20260927)
records = []

for sample_size in sample_sizes:
    for repetition in range(repetitions):
        estimated_mean, estimated_covariance = simulates_cov_mu(
            population_mean,
            population_covariance,
            n_obs=int(sample_size),
            shrink=False,
        )
        estimated_weights = minimum_variance_weights(estimated_covariance)
        estimated_risk = estimated_weights @ population_covariance @ estimated_weights
        records.append({
            "Sample size": int(sample_size),
            "Mean error": np.linalg.norm(estimated_mean - population_mean),
            "Relative covariance error": np.linalg.norm(
                estimated_covariance - population_covariance,
                ord="fro",
            ) / np.linalg.norm(population_covariance, ord="fro"),
            "Risk inflation": estimated_risk / minimum_risk - 1.0,
        })

simulation_results = pd.DataFrame(records)
finite_sample_summary = simulation_results.groupby("Sample size").agg(
    Median_mean_error=("Mean error", "median"),
    Median_relative_covariance_error=("Relative covariance error", "median"),
    Median_risk_inflation=("Risk inflation", "median"),
    Risk_inflation_90th_percentile=("Risk inflation", lambda values: values.quantile(0.90)),
).reset_index()
finite_sample_summary.columns = [
    "Sample size",
    "Median mean error",
    "Median relative covariance error",
    "Median risk inflation",
    "Risk inflation 90th percentile",
]
finite_sample_summary.to_csv(TABLES / "finite_sample_summary.csv", index=False)
finite_sample_latex = compact_table_latex(finite_sample_summary.to_latex(
    index=False,
    float_format=compact_float_format("%.6f"),
    caption="Monte Carlo errors and portfolio risk inflation by sample size.",
    label="tab:finite-sample-summary",
    position="tbp",
))
finite_sample_latex = finite_sample_latex.replace(
    r"\begin{tabular}",
    r"\begin{adjustbox}{max width=\textwidth}\begin{tabular}",
    1,
).replace(r"\end{tabular}", r"\end{tabular}\end{adjustbox}", 1)
(TABLES / "finite_sample_summary.tex").write_text(finite_sample_latex, encoding="utf-8")
display(finite_sample_summary.round(6))

,Sample size,Median mean error,Median relative covariance error,Median risk inflation,Risk inflation 90th percentile
0,40,0.038416,0.298577,0.118395,0.277343
1,80,0.027832,0.208046,0.056375,0.119623
2,160,0.019592,0.144740,0.026545,0.055976
3,320,0.013605,0.104970,0.014728,0.033104
4,640,0.009129,0.075271,0.006471,0.014065
5,1280,0.006953,0.051075,0.003509,0.006991


In [3]:
reference_rate = finite_sample_summary["Median mean error"].iloc[0] * np.sqrt(sample_sizes[0] / sample_sizes)
risk_quantiles = simulation_results.groupby("Sample size")["Risk inflation"].quantile([0.10, 0.50, 0.90]).unstack()
finite_sample_figure_data = pd.DataFrame({
    "Sample size": sample_sizes,
    "Median mean error": finite_sample_summary["Median mean error"].to_numpy(),
    "Median relative covariance error": finite_sample_summary["Median relative covariance error"].to_numpy(),
    "Inverse square-root guide": reference_rate,
    "Risk inflation 10th percentile": risk_quantiles[0.10].reindex(sample_sizes).to_numpy(),
    "Median risk inflation": risk_quantiles[0.50].reindex(sample_sizes).to_numpy(),
    "Risk inflation 90th percentile": risk_quantiles[0.90].reindex(sample_sizes).to_numpy(),
})
finite_sample_figure_data.to_csv(FIGURE_DATA / "finite_sample_convergence.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(10.5, 4.4), constrained_layout=True)
axes[0].loglog(
    sample_sizes,
    finite_sample_summary["Median mean error"],
    marker="o",
    color=palette["cyan"],
    linewidth=2.0,
    label="Mean estimation error",
)
axes[0].loglog(
    sample_sizes,
    finite_sample_summary["Median relative covariance error"],
    marker="s",
    color=palette["green"],
    linewidth=2.0,
    label="Relative covariance error",
)
axes[0].loglog(sample_sizes, reference_rate, color=palette["amber"], linewidth=1.6, linestyle=":", label="Inverse square-root guide")
axes[0].set_xlabel("Sample size")
axes[0].set_ylabel("Median error")
axes[0].set_title("Finite-Sample Estimation Error")
axes[0].legend()
axes[1].semilogx(sample_sizes, risk_quantiles[0.50], marker="o", color=palette["rose"], linewidth=2.0, label="Median")
axes[1].fill_between(sample_sizes, risk_quantiles[0.10], risk_quantiles[0.90], color=palette["rose"], alpha=0.24, label="10th to 90th percentiles")
axes[1].axhline(0.0, color=palette["amber"], linewidth=1.0)
axes[1].set_xlabel("Sample size")
axes[1].set_ylabel("Population risk inflation")
axes[1].set_title("Decision Uncertainty")
axes[1].legend()
figure.savefig(FIGURES / "finite_sample_convergence.png", bbox_inches="tight")
figure.savefig(FIGURES / "finite_sample_convergence.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

<Figure size 1050x440 with 2 Axes>

## Numerical Tolerance

The scaled error of a computed linear-system solution should be small relative to rounding error and problem difficulty:

$$
\frac{\lVert A\widehat x-b\rVert_2}{\lVert A\rVert_2\lVert\widehat x\rVert_2+\lVert b\rVert_2}
\leq C\,\varepsilon_{\mathrm{mach}}\,\kappa_2(A).
$$

Here, $A$ is the coefficient matrix; $\widehat x$ is the computed solution; $b$ is the right-hand side. $\lVert\cdot\rVert_2$ is the Euclidean or spectral norm; $C$ is a chosen safety factor; $\varepsilon_{\mathrm{mach}}$ is machine precision. $\kappa_2(A)$ is the condition number of $A$ in the spectral norm.

In [4]:
tolerance_generator = np.random.default_rng(20260928)
orthogonal_matrix, _ = np.linalg.qr(tolerance_generator.standard_normal((6, 6)))
condition_numbers = np.array([1.0e1, 1.0e3, 1.0e5, 1.0e7, 1.0e9])
right_hand_side = np.ones(6)
machine_epsilon = np.finfo(float).eps
tolerance_records = []

for condition_number in condition_numbers:
    eigenvalues = np.geomspace(1.0, 1.0 / condition_number, 6)
    matrix = orthogonal_matrix @ np.diag(eigenvalues) @ orthogonal_matrix.T
    solution = np.linalg.solve(matrix, right_hand_side)
    residual = np.linalg.norm(matrix @ solution - right_hand_side)
    scaled_residual = residual / (
        np.linalg.norm(matrix, ord=2) * np.linalg.norm(solution)
        + np.linalg.norm(right_hand_side)
    )
    tolerance = 100.0 * machine_epsilon * np.linalg.cond(matrix)
    tolerance_records.append({
        "Condition number": np.linalg.cond(matrix),
        "Scaled residual": scaled_residual,
        "Tolerance": tolerance,
    })

numerical_tolerance = pd.DataFrame(tolerance_records)
numerical_tolerance.to_csv(TABLES / "numerical_tolerance.csv", index=False)
numerical_tolerance.to_csv(FIGURE_DATA / "numerical_tolerance.csv", index=False)
tolerance_latex = compact_table_latex(numerical_tolerance.to_latex(
    index=False,
    float_format=compact_float_format("%.6e"),
    caption="Scale-sensitive numerical tolerances for linear systems with increasing condition numbers.",
    label="tab:numerical-tolerance",
    position="tbp",
))
tolerance_latex = tolerance_latex.replace(
    r"\begin{tabular}",
    r"\begin{adjustbox}{max width=\textwidth}\begin{tabular}",
    1,
).replace(r"\end{tabular}", r"\end{tabular}\end{adjustbox}", 1)
(TABLES / "numerical_tolerance.tex").write_text(tolerance_latex, encoding="utf-8")
display(numerical_tolerance)

figure, axis = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
plotted_residual = np.maximum(numerical_tolerance["Scaled residual"], np.finfo(float).tiny)
axis.loglog(numerical_tolerance["Condition number"], plotted_residual, marker="o", color=palette["cyan"], linewidth=2.0, label="Observed scaled residual")
axis.loglog(numerical_tolerance["Condition number"], numerical_tolerance["Tolerance"], marker="s", color=palette["amber"], linewidth=2.0, label="Scale-sensitive tolerance")
axis.set_xlabel("Matrix condition number")
axis.set_ylabel("Residual or tolerance")
axis.set_title("Numerical Error Depends on Problem Conditioning")
axis.legend()
figure.savefig(FIGURES / "numerical_tolerance.png", bbox_inches="tight")
figure.savefig(FIGURES / "numerical_tolerance.pdf", bbox_inches="tight")
display(figure)
plt.close(figure)

,Condition number,Scaled residual,Tolerance
0,10,0,0
1,1000,0,2.220446e-11
2,100000,0,2.220446e-09
3,10000000,0,2.220446e-07
4,1000000000,0,2.220446e-05


<Figure size 720x460 with 1 Axes>

## Tests

In [5]:
np.random.seed(271828)
reproducible_mean_a, reproducible_covariance_a = simulates_cov_mu(
    population_mean,
    population_covariance,
    n_obs=80,
    shrink=False,
)
np.random.seed(271828)
reproducible_mean_b, reproducible_covariance_b = simulates_cov_mu(
    population_mean,
    population_covariance,
    n_obs=80,
    shrink=False,
)
assert np.array_equal(reproducible_mean_a, reproducible_mean_b)
assert np.array_equal(reproducible_covariance_a, reproducible_covariance_b)
assert len(simulation_results) == len(sample_sizes) * repetitions
assert finite_sample_summary["Median mean error"].iloc[-1] < finite_sample_summary["Median mean error"].iloc[0]
assert finite_sample_summary["Median relative covariance error"].iloc[-1] < finite_sample_summary["Median relative covariance error"].iloc[0]
assert finite_sample_summary["Median risk inflation"].iloc[-1] < finite_sample_summary["Median risk inflation"].iloc[0]
assert np.all(simulation_results["Risk inflation"] >= -1.0e-10)
assert np.all(numerical_tolerance["Scaled residual"] <= numerical_tolerance["Tolerance"])
assert np.all(np.diff(numerical_tolerance["Condition number"]) > 0.0)